### 슬기로운 대학원 생활
#### 1. 에이전트 이름
슬기로운 대학원 생활(논문 읽기 어시스턴트)

#### 2. 에이전트 목적
영어를 못하는 상황에서 해외 논문을 읽는건 정말 어려웠다. 그렇다고 번역기만 돌리자니 대학원을 다니고 있는 학생이 맞는건가 싶은 자괴감이 들었다. 어차피 논문을 쓰려면 AI를 사용한다고 하더라도 내가 논문 내용을 검수해야하고, 그러려면 영어 능력도 높여야 한다. 다만, 논문을 읽다보면 참고 문헌의 어떤 부분을 참고 했는지 보고 싶어도 찾는데 시간이 꽤 걸리고, 어려운 영어 어휘나 표현이 나오면 그걸 찾다가 논문을 읽기 싫어지기도 한다. 그래서 한 번에 논문을 읽으면서 개인 역량도 키울 수 있는 에이전트가 있으면 좋겠다고 생각했다.

#### 3. 핵심 기능
1. PDF 파일의 텍스트 및 자료 인식 기능
2. 텍스트에서 영어 어휘 및 표현 추출 기능 + 단어장 같은 저장 기능
3. 참고 자료에 대한 링크를 찾거나, 인용한 문구 부분까지 찾아주는 기능.

In [1]:
from dotenv import load_dotenv
load_dotenv()

import sqlite3
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.sqlite import SqliteSaver
from typing_extensions import TypedDict

llm = init_chat_model("openai:gpt-4o")

# conn = sqlite3.connect("memory.db", check_same_thread=False)

# config={
#     "configurable": {
#         "thread_id": "test-260326",
#     },
#     "recursion_limit": 2,
# }

In [2]:
class ContentState(TypedDict):
    content: str
    vocas: list[str]

class InputState(TypedDict):
    title: str
    content: str

class OutputState(TypedDict):
    vocas: list[str]
    refs: list[str]

graph_builder = StateGraph(
    ContentState,
    input_schema=InputState,
    output_schema=OutputState,
)

In [3]:
def extract_content(state: InputState) -> ContentState:
    return {
        "content": state["content"]
    }

def extract_vocabulary(state: ContentState) -> OutputState:
    response = llm.invoke(f"""
    You have a goode english skills.
    After read content below, extract 10 words with korean mean.

    {state["content"]}
    """)
    return { "vocas": response }


In [4]:
graph_builder.add_node("extract_content", extract_content)
graph_builder.add_node("extract_vocabulary", extract_vocabulary)

graph_builder.add_edge(START, "extract_content")
graph_builder.add_edge("extract_content", "extract_vocabulary")
graph_builder.add_edge("extract_vocabulary", END)

graph = graph_builder.compile(
    # checkpointer=SqliteSaver(conn)
)

In [7]:
result = graph.invoke(
    {
        "title": "MSCRS: Multi-modal Semantic Graph Prompt Learning Framework for Conversational Recommender Systems",
        "content": """
Conversational Recommender Systems (CRSs) aim to provide per-
sonalized recommendations by interacting with users through
conversations. Most existing studies of CRS focus on extracting
user preferences from conversational contexts. However, due to
the short and sparse nature of conversational contexts, it is diffi-
cult to fully capture user preferences by conversational contexts
only. We argue that multi-modal semantic information can en-
rich user preference expressions from diverse dimensions (e.g., a
user preference for a certain movie may stem from its magnif-
icent visual effects and compelling storyline). In this paper, we
propose a multi-modal semantic graph prompt learning framework
for CRS, named MSCRS. First, we extract textual and image fea-
tures of items mentioned in the conversational contexts. Second, we
capture higher-order semantic associations within different seman-
tic modalities (collaborative, textual, and image) by constructing
modality-specific graph structures. Finally, we propose an inno-
vative integration of multi-modal semantic graphs with prompt
learning, harnessing the power of large language models to compre-
hensively explore high-dimensional semantic relationships. Experi-
mental results demonstrate that our proposed method significantly
improves accuracy in item recommendation, as well as generates
more natural and contextually relevant content in response genera-
tion. Code and extended multi-modal CRS datasets are available at
https://github.com/BIAOBIAO12138/MSCRS-main
        """
    },
    # config= config
)

print(result)

{'vocas': AIMessage(content='Here are 10 words from the text along with their Korean meanings:\n\n1. Conversational (회화의)\n2. Recommender (추천 시스템)\n3. Personalized (개인화된)\n4. Preferences (선호도)\n5. Semantic (의미론적인)\n6. Visual (시각적인)\n7. Framework (체계)\n8. Features (특징)\n9. Accuracy (정확성)\n10. Response (응답)', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 335, 'total_tokens': 430, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_8deb0dfbd4', 'id': 'chatcmpl-DNPRNmTXv1LDDZEzG1mAWPsepkbnT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d26c1-80a7-7823-be03-f8cc8488d65e-0', usage_metadata={'input_tokens': 335, 'output_tokens': 95, 'total_tokens': 430, 'input_token_details': 